In [4]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path
import matplotlib
matplotlib.use('Agg') 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from tqdm import tqdm
import numpy as np

# ================= 配置 =================
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
DATASET_ROOT = "./dataset_zero_cfo_rad" 

SIM_TRAIN_DIR = os.path.join(DATASET_ROOT, "sim_train")
SIM_VAL_DIR = os.path.join(DATASET_ROOT, "sim_val")
REAL_TEST_DIR = os.path.join(DATASET_ROOT, "real_test")

RESULTS_DIR = "./results_zero_cfo" 
os.makedirs(RESULTS_DIR, exist_ok=True)

CLASS_NAMES = [
    "BPSK", "QPSK", "8PSK", "OQPSK", "PI4DQPSK", 
    "16QAM", "32QAM", "64QAM", 
    "2ASK", "16APSK", "32APSK", "FSK2"
]
NUM_CLASSES = len(CLASS_NAMES)
IMAGE_SIZE = 64
BATCH_SIZE = 128
LEARNING_RATE = 0.001
NUM_EPOCHS = 50

# ================= 1. 数据集 & 模型 =================
class InMemoryDataset(Dataset):
    def __init__(self, root_dir, has_snr=False):
        self.data = []
        self.labels = []
        self.snrs = []
        self.transform = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor()])
        
        if not os.path.exists(root_dir): return
        print(f"Loading {root_dir}...")
        files = []
        for cls in CLASS_NAMES:
            d = Path(root_dir) / cls
            if d.exists(): files.extend([(p, CLASS_NAMES.index(cls)) for p in d.glob("*.png")])
            
        with tqdm(total=len(files), unit="img") as pbar:
            for p, label in files:
                try:
                    snr = -999.0
                    if has_snr:
                        try: snr = float(p.stem.split('_')[-1])
                        except: pass
                    with Image.open(str(p)).convert("RGB") as img:
                        self.data.append(self.transform(img))
                        self.labels.append(label)
                        self.snrs.append(snr)
                except: pass
                pbar.update(1)
    
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx], self.labels[idx], self.snrs[idx]

class TinyCNN(nn.Module):
    def __init__(self, num_classes):
        super(TinyCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout2d(0.1),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout2d(0.1),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8, 128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )
    def forward(self, x): return self.classifier(self.features(x))

def plot_confusion_matrix(y_true, y_pred, title, save_path):
    present_indices = sorted(list(set(y_true) | set(y_pred)))
    present_names = [CLASS_NAMES[i] for i in present_indices]
    cm = confusion_matrix(y_true, y_pred, labels=present_indices)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=present_names, yticklabels=present_names)
    plt.title(title); plt.ylabel('True'); plt.xlabel('Pred')
    plt.tight_layout(); plt.savefig(save_path); plt.close()

# ================= 4. 主程序 =================
def main():
    train_ds = InMemoryDataset(SIM_TRAIN_DIR, has_snr=False)
    val_ds = InMemoryDataset(SIM_VAL_DIR, has_snr=True)
    real_ds = InMemoryDataset(REAL_TEST_DIR, has_snr=False)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    model = TinyCNN(NUM_CLASSES).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc = 0.0
    
    print(f"\nStart Training (Zero CFO Sim)...")
    for epoch in range(NUM_EPOCHS):
        model.train()
        r_loss, correct, total = 0.0, 0, 0
        for imgs, labels, _ in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            r_loss += loss.item() * imgs.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
        t_loss = r_loss/total; t_acc = correct/total
        
        model.eval()
        v_r_loss, v_correct, v_total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels, _ in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                v_r_loss += loss.item() * imgs.size(0)
                _, preds = outputs.max(1)
                v_correct += preds.eq(labels).sum().item()
                v_total += labels.size(0)
        v_loss = v_r_loss/v_total; v_acc = v_correct/v_total
        
        history['train_loss'].append(t_loss); history['val_loss'].append(v_loss)
        history['train_acc'].append(t_acc); history['val_acc'].append(v_acc)
        
        log = f"Epoch {epoch+1} | Loss: {t_loss:.4f}/{v_loss:.4f} | Acc: {t_acc:.4f}/{v_acc:.4f}"
        if v_acc > best_val_acc:
            best_val_acc = v_acc
            torch.save(model.state_dict(), os.path.join(RESULTS_DIR, "best_model_zero_cfo.pth"))
            log += " [Best!]"
        print(log)

    # Plot Curves
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1); plt.plot(history['train_loss'], label='Train'); plt.plot(history['val_loss'], label='Val')
    plt.title('Loss'); plt.legend()
    plt.subplot(1, 2, 2); plt.plot(history['train_acc'], label='Train'); plt.plot(history['val_acc'], label='Val')
    plt.title('Acc'); plt.legend()
    plt.savefig(os.path.join(RESULTS_DIR, "curves.png")); plt.close()

    # --- Final Test on Real Data ---
    print("\n[TEST] Real Data Evaluation...")
    if len(real_ds) > 0:
        model.load_state_dict(torch.load(os.path.join(RESULTS_DIR, "best_model_zero_cfo.pth")))
        model.eval()
        real_loader = DataLoader(real_ds, batch_size=BATCH_SIZE)
        y_true, y_pred = [], []
        with torch.no_grad():
            for imgs, labels, _ in real_loader:
                outputs = model(imgs.to(DEVICE))
                _, preds = outputs.max(1)
                y_true.extend(labels.cpu().numpy())
                y_pred.extend(preds.cpu().numpy())
        
        acc = np.mean(np.array(y_true) == np.array(y_pred))
        print(f">>> Real Data Accuracy: {acc:.2%}")
        plot_confusion_matrix(y_true, y_pred, f"Real Data (Acc: {acc:.2%})", 
                              os.path.join(RESULTS_DIR, "cm_real.png"))
    else:
        print("No real data found.")

if __name__ == "__main__":
    main()

Loading ./dataset_zero_cfo_rad\sim_train...


100%|██████████| 7689/7689 [00:04<00:00, 1811.21img/s]


Loading ./dataset_zero_cfo_rad\sim_val...


100%|██████████| 1911/1911 [00:00<00:00, 3183.13img/s]


Loading ./dataset_zero_cfo_rad\real_test...


100%|██████████| 43938/43938 [00:11<00:00, 3953.69img/s]



Start Training (Zero CFO Sim)...
Epoch 1 | Loss: 1.6835/0.8583 | Acc: 0.3841/0.7823 [Best!]
Epoch 2 | Loss: 0.8249/0.4017 | Acc: 0.6832/0.8760 [Best!]
Epoch 3 | Loss: 0.5439/0.2878 | Acc: 0.7907/0.9037 [Best!]
Epoch 4 | Loss: 0.4386/0.2686 | Acc: 0.8253/0.8964
Epoch 5 | Loss: 0.4089/0.2425 | Acc: 0.8404/0.9058 [Best!]
Epoch 6 | Loss: 0.3670/0.2601 | Acc: 0.8510/0.8891
Epoch 7 | Loss: 0.3448/0.2208 | Acc: 0.8624/0.9100 [Best!]
Epoch 8 | Loss: 0.3108/0.1721 | Acc: 0.8718/0.9299 [Best!]
Epoch 9 | Loss: 0.2815/0.1666 | Acc: 0.8872/0.9273
Epoch 10 | Loss: 0.2699/0.1813 | Acc: 0.8879/0.9168
Epoch 11 | Loss: 0.2654/0.1382 | Acc: 0.8887/0.9393 [Best!]
Epoch 12 | Loss: 0.2626/0.1695 | Acc: 0.8869/0.9273
Epoch 13 | Loss: 0.2595/0.1394 | Acc: 0.8944/0.9383
Epoch 14 | Loss: 0.2253/0.1284 | Acc: 0.9075/0.9409 [Best!]
Epoch 15 | Loss: 0.2222/0.1480 | Acc: 0.9094/0.9341
Epoch 16 | Loss: 0.2155/0.1265 | Acc: 0.9078/0.9403
Epoch 17 | Loss: 0.2229/0.1267 | Acc: 0.9061/0.9424 [Best!]
Epoch 18 | Loss: 0.